In [ ]:
import os
import sys
from tempfile import NamedTemporaryFile
from urllib.request import urlopen
from urllib.parse import unquote, urlparse
from urllib.error import HTTPError
from zipfile import ZipFile
import tarfile
import shutil

CHUNK_SIZE = 40960
DATA_SOURCE_MAPPING = 'agriculture-crop-images:https%3A%2F%2Fstorage.googleapis.com%2Fkaggle-data-sets%2F775339%2F2011301%2Fbundle%2Farchive.zip%3FX-Goog-Algorithm%3DGOOG4-RSA-SHA256%26X-Goog-Credential%3Dgcp-kaggle-com%2540kaggle-161607.iam.gserviceaccount.com%252F20240420%252Fauto%252Fstorage%252Fgoog4_request%26X-Goog-Date%3D20240420T075849Z%26X-Goog-Expires%3D259200%26X-Goog-SignedHeaders%3Dhost%26X-Goog-Signature%3D949e0e8b4d34934172b100466ea799c096da7880793f5b2b240e246cd03b1dd141c91bc6f14613fea1863ec4c26d5c8c0c68f6ffb58434c047b1cd48c078a12f9b01043a9703da6ead4a030b299c6570c19b1fde83877852a7d40050652518de8b7358550f56031f292d4cb03f41bb010f5796a34dbebac5ca933841d18167b71374361fc9cf54e8edeb871d5599189904b452f917c0bb45ff63c7c3723497f8cd23cca5ce1b27f9e8a26a974f475293c6a238a3486449fcd5e4c679217cbb9be276cc324572bb736d5868fde780fd74d33b0630aee3f979a2c4042e5092e1cc114e77ec9819cfdaf6c1f17791ce5f0201b42d3bf3533e544057414954d8b822'

KAGGLE_INPUT_PATH='/kaggle/input'
KAGGLE_WORKING_PATH='/kaggle/working'
KAGGLE_SYMLINK='kaggle'

!umount /kaggle/input/ 2> /dev/null
shutil.rmtree('/kaggle/input', ignore_errors=True)
os.makedirs(KAGGLE_INPUT_PATH, 0o777, exist_ok=True)
os.makedirs(KAGGLE_WORKING_PATH, 0o777, exist_ok=True)

try:
  os.symlink(KAGGLE_INPUT_PATH, os.path.join("..", 'input'), target_is_directory=True)
except FileExistsError:
  pass
try:
  os.symlink(KAGGLE_WORKING_PATH, os.path.join("..", 'working'), target_is_directory=True)
except FileExistsError:
  pass

for data_source_mapping in DATA_SOURCE_MAPPING.split(','):
    directory, download_url_encoded = data_source_mapping.split(':')
    download_url = unquote(download_url_encoded)
    filename = urlparse(download_url).path
    destination_path = os.path.join(KAGGLE_INPUT_PATH, directory)
    try:
        with urlopen(download_url) as fileres, NamedTemporaryFile() as tfile:
            total_length = fileres.headers['content-length']
            print(f'Downloading {directory}, {total_length} bytes compressed')
            dl = 0
            data = fileres.read(CHUNK_SIZE)
            while len(data) > 0:
                dl += len(data)
                tfile.write(data)
                done = int(50 * dl / int(total_length))
                sys.stdout.write(f"\r[{'=' * done}{' ' * (50-done)}] {dl} bytes downloaded")
                sys.stdout.flush()
                data = fileres.read(CHUNK_SIZE)
            if filename.endswith('.zip'):
              with ZipFile(tfile) as zfile:
                zfile.extractall(destination_path)
            else:
              with tarfile.open(tfile.name) as tarfile:
                tarfile.extractall(destination_path)
            print(f'\nDownloaded and uncompressed: {directory}')
    except HTTPError as e:
        print(f'Failed to load (likely expired) {download_url} to path {destination_path}')
        continue
    except OSError as e:
        print(f'Failed to load {download_url} to path {destination_path}')
        continue

print('Data source import complete.')


[==================================================] 62575817 bytes downloaded
Downloaded and uncompressed: agriculture-crop-images
Data source import complete.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
data_gen = ImageDataGenerator(zoom_range=0.2, rotation_range=0.2, horizontal_flip=True, rescale=1/255)

In [ ]:
train_data_1 = data_gen.flow_from_directory(directory="/kaggle/input/agriculture-crop-images/crop_images", target_size=(224, 224))

Found 201 images belonging to 5 classes.


In [ ]:
train_data_1.class_indices

{'jute': 0, 'maize': 1, 'rice': 2, 'sugarcane': 3, 'wheat': 4}

In [ ]:
train_data_2 = data_gen.flow_from_directory(directory="/kaggle/input/agriculture-crop-images/kag2", target_size=(224, 224))

Found 804 images belonging to 5 classes.


In [ ]:
train_data_2.class_indices

{'jute': 0, 'maize': 1, 'rice': 2, 'sugarcane': 3, 'wheat': 4}

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import legacy

optimizer = Adam(learning_rate=0.001)

# You can adjust the learning rate as needed

optimizer_legacy = legacy.Adam(learning_rate=0.001)


In [ ]:
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Flatten
from tensorflow.keras.models import Sequential

model = Sequential()
model.add(Conv2D(32, (3, 3), activation = 'relu', input_shape = (224, 224, 3)))
model.add(MaxPool2D())
model.add(Conv2D(64, (3, 3), activation = 'relu'))
model.add(MaxPool2D())
model.add(Conv2D(128, (3, 3), activation = 'relu'))
model.add(MaxPool2D())
model.add(Conv2D(128, (5, 5), activation = 'relu'))
model.add(MaxPool2D())
model.add(Conv2D(256, (3, 3), activation = 'relu'))
model.add(MaxPool2D())
model.add(Flatten())
model.add(Dense(5, activation = 'softmax'))

model.compile(optimizer=optimizer,loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_99 (Conv2D)          (None, 222, 222, 32)      896       
                                                                 
 max_pooling2d_9 (MaxPoolin  (None, 111, 111, 32)      0         
 g2D)                                                            
                                                                 
 conv2d_100 (Conv2D)         (None, 109, 109, 64)      18496     
                                                                 
 max_pooling2d_10 (MaxPooli  (None, 54, 54, 64)        0         
 ng2D)                                                           
                                                                 
 conv2d_101 (Conv2D)         (None, 52, 52, 128)       73856     
                                                                 
 max_pooling2d_11 (MaxPooli  (None, 26, 26, 128)      

In [ ]:
model.fit(train_data_1, epochs=10)

Epoch 1/10
7/7 [==============================] - 4s 404ms/step - loss: 1.6157 - accuracy: 0.1741
Epoch 2/10
7/7 [==============================] - 3s 394ms/step - loss: 1.6078 - accuracy: 0.2438
Epoch 3/10
7/7 [==============================] - 3s 391ms/step - loss: 1.5778 - accuracy: 0.2687
Epoch 4/10
7/7 [==============================] - 3s 396ms/step - loss: 1.4577 - accuracy: 0.3731
Epoch 5/10
7/7 [==============================] - 3s 389ms/step - loss: 1.4367 - accuracy: 0.4129
Epoch 6/10
7/7 [==============================] - 3s 393ms/step - loss: 1.3230 - accuracy: 0.4577
Epoch 7/10
7/7 [==============================] - 3s 388ms/step - loss: 1.4416 - accuracy: 0.4080
Epoch 8/10
7/7 [==============================] - 3s 395ms/step - loss: 1.3581 - accuracy: 0.4229
Epoch 9/10
7/7 [==============================] - 3s 390ms/step - loss: 1.3211 - accuracy: 0.4776
Epoch 10/10
7/7 [==============================] - 3s 463ms/step - loss: 1.2486 - accuracy: 0.4378


In [ ]:
model.fit(train_data_2, epochs=10)

Epoch 1/10
26/26 [==============================] - 11s 421ms/step - loss: 1.2325 - accuracy: 0.4801
Epoch 2/10
26/26 [==============================] - 11s 422ms/step - loss: 1.2700 - accuracy: 0.4627
Epoch 3/10
26/26 [==============================] - 11s 425ms/step - loss: 1.1714 - accuracy: 0.5311
Epoch 4/10
26/26 [==============================] - 11s 418ms/step - loss: 1.0726 - accuracy: 0.5522
Epoch 5/10
26/26 [==============================] - 12s 430ms/step - loss: 0.9681 - accuracy: 0.6256
Epoch 6/10
26/26 [==============================] - 11s 425ms/step - loss: 0.9045 - accuracy: 0.6480
Epoch 7/10
26/26 [==============================] - 11s 419ms/step - loss: 0.9636 - accuracy: 0.6107
Epoch 8/10
26/26 [==============================] - 11s 425ms/step - loss: 0.8520 - accuracy: 0.6841
Epoch 9/10
26/26 [==============================] - 11s 423ms/step - loss: 0.7256 - accuracy: 0.7002
Epoch 10/10
26/26 [==============================] - 11s 427ms/step - loss: 0.6559 - accura

In [ ]:
model.save('agri_crops_1.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16

In [ ]:
vgg16 = VGG16(include_top=False,input_shape=(224, 224, 3))
vgg16.summary()

58889256/58889256 [==============================] - 0s 0us/step
Model: "vgg16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                              

In [ ]:
for layer in vgg16.layers:
    layer.trainable = False

vgg16.summary()

Model: "vgg16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [ ]:
last_layer = Dense(5, activation='softmax')(Flatten()(vgg16.output))

In [ ]:
from tensorflow.keras.models import Model

model_vgg = Model(inputs = vgg16.input, outputs = last_layer)
model_vgg.compile(optimizer=optimizer_legacy,loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model_vgg.fit(train_data_1, epochs=10)

Epoch 1/10
7/7 [==============================] - 5s 512ms/step - loss: 1.9922 - accuracy: 0.2687
Epoch 2/10
7/7 [==============================] - 4s 517ms/step - loss: 1.3906 - accuracy: 0.4378
Epoch 3/10
7/7 [==============================] - 4s 526ms/step - loss: 1.0070 - accuracy: 0.6020
Epoch 4/10
7/7 [==============================] - 4s 621ms/step - loss: 0.7578 - accuracy: 0.7264
Epoch 5/10
7/7 [==============================] - 4s 509ms/step - loss: 0.5903 - accuracy: 0.8159
Epoch 6/10
7/7 [==============================] - 4s 522ms/step - loss: 0.5187 - accuracy: 0.8408
Epoch 7/10
7/7 [==============================] - 4s 521ms/step - loss: 0.5245 - accuracy: 0.8308
Epoch 8/10
7/7 [==============================] - 4s 533ms/step - loss: 0.3913 - accuracy: 0.9154
Epoch 9/10
7/7 [==============================] - 4s 514ms/step - loss: 0.3389 - accuracy: 0.9303
Epoch 10/10
7/7 [==============================] - 4s 518ms/step - loss: 0.2988 - accuracy: 0.9353


In [ ]:
model_vgg.fit(train_data_2, epochs=10)

Epoch 1/10
26/26 [==============================] - 15s 567ms/step - loss: 0.2723 - accuracy: 0.9453
Epoch 2/10
26/26 [==============================] - 15s 562ms/step - loss: 0.1935 - accuracy: 0.9701
Epoch 3/10
26/26 [==============================] - 15s 550ms/step - loss: 0.1448 - accuracy: 0.9851
Epoch 4/10
26/26 [==============================] - 15s 558ms/step - loss: 0.1159 - accuracy: 0.9938
Epoch 5/10
26/26 [==============================] - 15s 555ms/step - loss: 0.0986 - accuracy: 0.9963
Epoch 6/10
26/26 [==============================] - 15s 559ms/step - loss: 0.0811 - accuracy: 0.9938
Epoch 7/10
26/26 [==============================] - 15s 560ms/step - loss: 0.0736 - accuracy: 0.9988
Epoch 8/10
26/26 [==============================] - 15s 593ms/step - loss: 0.0625 - accuracy: 0.9988
Epoch 9/10
26/26 [==============================] - 15s 555ms/step - loss: 0.0542 - accuracy: 0.9988
Epoch 10/10
26/26 [==============================] - 15s 566ms/step - loss: 0.0432 - accura

In [ ]:
model_vgg.save('agri_crops_vgg.h5')

In [ ]:
from tensorflow.keras.applications.resnet50 import ResNet50

resnet50 = ResNet50(include_top = False, input_shape = (224, 224, 3))

for layer in resnet50.layers:
    layer.trainable = False

last_layer = Dense(5, activation='softmax')(Flatten()(resnet50.output))

model_resnet = Model(inputs = resnet50.input, outputs = last_layer)
model_resnet.compile(optimizer=optimizer_legacy,loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model_resnet.fit(train_data_1, epochs=10)

Epoch 1/10
7/7 [==============================] - 6s 576ms/step - loss: 23.3169 - accuracy: 0.2239
Epoch 2/10
7/7 [==============================] - 4s 501ms/step - loss: 11.3259 - accuracy: 0.2338
Epoch 3/10
7/7 [==============================] - 4s 487ms/step - loss: 8.6384 - accuracy: 0.1891
Epoch 4/10
7/7 [==============================] - 4s 576ms/step - loss: 6.3076 - accuracy: 0.2040
Epoch 5/10
7/7 [==============================] - 4s 483ms/step - loss: 4.4875 - accuracy: 0.2935
Epoch 6/10
7/7 [==============================] - 4s 470ms/step - loss: 4.5098 - accuracy: 0.3035
Epoch 7/10
7/7 [==============================] - 4s 497ms/step - loss: 4.1266 - accuracy: 0.2985
Epoch 8/10
7/7 [==============================] - 4s 476ms/step - loss: 3.1554 - accuracy: 0.3035
Epoch 9/10
7/7 [==============================] - 4s 494ms/step - loss: 2.6851 - accuracy: 0.3881
Epoch 10/10
7/7 [==============================] - 4s 480ms/step - loss: 2.4961 - accuracy: 0.3085


In [ ]:
model_resnet.fit(train_data_2, epochs=10)

Epoch 1/10
26/26 [==============================] - 14s 526ms/step - loss: 2.1143 - accuracy: 0.3818
Epoch 2/10
26/26 [==============================] - 14s 527ms/step - loss: 2.0481 - accuracy: 0.3856
Epoch 3/10
26/26 [==============================] - 14s 530ms/step - loss: 1.8452 - accuracy: 0.4391
Epoch 4/10
26/26 [==============================] - 14s 527ms/step - loss: 1.6244 - accuracy: 0.4565
Epoch 5/10
26/26 [==============================] - 14s 553ms/step - loss: 1.9303 - accuracy: 0.4540
Epoch 6/10
26/26 [==============================] - 14s 523ms/step - loss: 1.1946 - accuracy: 0.5510
Epoch 7/10
26/26 [==============================] - 14s 524ms/step - loss: 1.1499 - accuracy: 0.5721
Epoch 8/10
26/26 [==============================] - 14s 525ms/step - loss: 1.2864 - accuracy: 0.5398
Epoch 9/10
26/26 [==============================] - 14s 525ms/step - loss: 1.3016 - accuracy: 0.5485
Epoch 10/10
26/26 [==============================] - 14s 525ms/step - loss: 1.3850 - accura

In [ ]:
model_resnet.save('agri_crops_resnet.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
from tensorflow.keras.applications.inception_v3 import InceptionV3

inceptionv3 = InceptionV3(include_top = False, input_shape = (224, 224, 3))

for layer in inceptionv3.layers:
    layer.trainable = False

last_layer = Dense(5, activation='softmax')(Flatten()(inceptionv3.output))

model_inception = Model(inputs = inceptionv3.input, outputs = last_layer)
model_inception.compile(optimizer=optimizer_legacy,loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model_inception.fit(train_data_1, epochs=10)

Epoch 1/10
7/7 [==============================] - 7s 360ms/step - loss: 37.7613 - accuracy: 0.2338
Epoch 2/10
7/7 [==============================] - 3s 355ms/step - loss: 15.1648 - accuracy: 0.4378
Epoch 3/10
7/7 [==============================] - 3s 363ms/step - loss: 11.8540 - accuracy: 0.5373
Epoch 4/10
7/7 [==============================] - 3s 362ms/step - loss: 6.4605 - accuracy: 0.7313
Epoch 5/10
7/7 [==============================] - 3s 403ms/step - loss: 4.6845 - accuracy: 0.8060
Epoch 6/10
7/7 [==============================] - 3s 356ms/step - loss: 4.0296 - accuracy: 0.7960
Epoch 7/10
7/7 [==============================] - 3s 358ms/step - loss: 3.0526 - accuracy: 0.8010
Epoch 8/10
7/7 [==============================] - 3s 360ms/step - loss: 2.0448 - accuracy: 0.8259
Epoch 9/10
7/7 [==============================] - 3s 356ms/step - loss: 2.2657 - accuracy: 0.8507
Epoch 10/10
7/7 [==============================] - 3s 358ms/step - loss: 1.3213 - accuracy: 0.8806


In [ ]:
model_inception.fit(train_data_2, epochs=10)

Epoch 1/10
26/26 [==============================] - 10s 380ms/step - loss: 1.1636 - accuracy: 0.9092
Epoch 2/10
26/26 [==============================] - 10s 391ms/step - loss: 0.3838 - accuracy: 0.9627
Epoch 3/10
26/26 [==============================] - 10s 387ms/step - loss: 0.1557 - accuracy: 0.9851
Epoch 4/10
26/26 [==============================] - 10s 387ms/step - loss: 0.2134 - accuracy: 0.9764
Epoch 5/10
26/26 [==============================] - 10s 389ms/step - loss: 0.2087 - accuracy: 0.9714
Epoch 6/10
26/26 [==============================] - 10s 389ms/step - loss: 0.1186 - accuracy: 0.9851
Epoch 7/10
26/26 [==============================] - 10s 379ms/step - loss: 0.1413 - accuracy: 0.9813
Epoch 8/10
26/26 [==============================] - 10s 384ms/step - loss: 0.0965 - accuracy: 0.9826
Epoch 9/10
26/26 [==============================] - 10s 390ms/step - loss: 0.0869 - accuracy: 0.9751
Epoch 10/10
26/26 [==============================] - 10s 390ms/step - loss: 0.1341 - accura

In [ ]:
model_inception.save('agri_crops_inception.h5')

### Predictions

In [ ]:
import cv2

In [ ]:
def get_class_name(filename):
    if filename[:4] == 'jute':
        return 'jute'
    elif filename[:5] == 'maize':
        return 'maize'
    elif filename[:4] == 'rice':
        return 'rice'
    elif filename[:9] == 'sugarcane':
        return 'sugarcane'
    elif filename[:5] == 'wheat':
        return 'wheat'
    else:
        return 'none'

In [ ]:
data = ['jute', 'maize', 'rice', 'sugarcane', 'wheat']

In [ ]:
def get_pred(filename, m):
    img = cv2.imread('/kaggle/input/agriculture-crop-images/test_crop_image/' + filename)
    img = cv2.resize(img, (224, 224))/255
    img = img.reshape(1, 224, 224, 3)
    yp = m.predict(img).argmax()
    return yp

In [ ]:
filenames = os.listdir('/kaggle/input/agriculture-crop-images/test_crop_image')

In [ ]:
filenames

['maize corn set2.jpg',
 'wheat-field.jfif',
 'wheatcropfield04.jpg',
 'maize00corn-fields.jpg',
 'maize-Field-Corn.jpg',
 'maizecornleaves.jfif',
 'rice-5174887_1280.jpg',
 'maize-field01.jpg',
 'juteimg.jpg',
 'rice-828540_1280.jpg',
 'wheat-crop.jpg',
 'jute002.jpg',
 'jute-field.jpg',
 'jute03.jpg',
 'maize plant set.jpg',
 'rice-field.jpg',
 'wheatss.jpg',
 'wheat-field03.jpg',
 'sugarcane-field.jpg',
 'wheatcrop01.jpg',
 'rice8122f869e3f.jpg',
 'sugarcane-farm-in-the-mountain-countryside-of-thailand.jpg',
 'jutefield.jpg',
 'rice-fields-204128_1280.jpg',
 'sugarcane-field8.jpg',
 'wheat-field01.jpg',
 'maize-field.jpg',
 'maize_fieldmexico.jpeg',
 'wheatcropfield.jpg',
 'wheat.jpg',
 'juteplant.jpg',
 'wheatcrops.jpg',
 'maize000.jfif',
 'juteleaves.jpg',
 'sugarcaneplants.jpg',
 'jute003.jpg',
 'wheat-field02.jpg',
 'rice-field01.jpg',
 'sugarcanefield.jpg',
 'wheat-field-artificial-irrigation-rural-electrification-to-harvest-166395991.jpg',
 'maize images.jfif',
 'maize02.jfif'

In [ ]:
preds_model = []
for file in filenames:
    yp = data[get_pred(file, model)]
    ya = get_class_name(file)
    pred = (yp == ya)
    preds_model.append(pred)
    print(file, yp, ya, pred)

1/1 [==============================] - 0s 33ms/step
maize corn set2.jpg rice maize False
1/1 [==============================] - 0s 31ms/step
wheat-field.jfif maize wheat False
1/1 [==============================] - 0s 30ms/step
wheatcropfield04.jpg rice wheat False
1/1 [==============================] - 0s 32ms/step
maize00corn-fields.jpg maize maize True
1/1 [==============================] - 0s 32ms/step
maize-Field-Corn.jpg rice maize False
1/1 [==============================] - 0s 34ms/step
maizecornleaves.jfif rice maize False
1/1 [==============================] - 0s 32ms/step
rice-5174887_1280.jpg sugarcane rice False
1/1 [==============================] - 0s 31ms/step
maize-field01.jpg rice maize False
1/1 [==============================] - 0s 31ms/step
juteimg.jpg rice jute False
1/1 [==============================] - 0s 32ms/step
rice-828540_1280.jpg sugarcane rice False
1/1 [==============================] - 0s 33ms/step
wheat-crop.jpg rice wheat False
1/1 [=================

In [ ]:
acc_model = sum(preds_model)/len(preds_model)
acc_model

0.17647058823529413

In [ ]:
preds_vgg = []
for file in filenames:
    yp = data[get_pred(file, model_vgg)]
    ya = get_class_name(file)
    pred = (yp == ya)
    preds_vgg.append(pred)
    print(file, yp, ya, pred)

1/1 [==============================] - 0s 72ms/step
maize corn set2.jpg maize maize True
1/1 [==============================] - 0s 69ms/step
wheat-field.jfif maize wheat False
1/1 [==============================] - 0s 70ms/step
wheatcropfield04.jpg sugarcane wheat False
1/1 [==============================] - 0s 70ms/step
maize00corn-fields.jpg maize maize True
1/1 [==============================] - 0s 70ms/step
maize-Field-Corn.jpg wheat maize False
1/1 [==============================] - 0s 71ms/step
maizecornleaves.jfif maize maize True
1/1 [==============================] - 0s 70ms/step
rice-5174887_1280.jpg maize rice False
1/1 [==============================] - 0s 72ms/step
maize-field01.jpg maize maize True
1/1 [==============================] - 0s 73ms/step
juteimg.jpg rice jute False
1/1 [==============================] - 0s 70ms/step
rice-828540_1280.jpg rice rice True
1/1 [==============================] - 0s 72ms/step
wheat-crop.jpg sugarcane wheat False
1/1 [================

In [ ]:
acc_vgg = sum(preds_vgg)/len(preds_vgg)
acc_vgg

0.49019607843137253

In [ ]:
preds_resnet = []
for file in filenames:
    yp = data[get_pred(file, model_resnet)]
    ya = get_class_name(file)
    pred = (yp == ya)
    preds_resnet.append(pred)
    print(file, yp, ya, pred)

1/1 [==============================] - 1s 913ms/step
maize corn set2.jpg wheat maize False
1/1 [==============================] - 0s 84ms/step
wheat-field.jfif rice wheat False
1/1 [==============================] - 0s 82ms/step
wheatcropfield04.jpg rice wheat False
1/1 [==============================] - 0s 82ms/step
maize00corn-fields.jpg maize maize True
1/1 [==============================] - 0s 82ms/step
maize-Field-Corn.jpg sugarcane maize False
1/1 [==============================] - 0s 83ms/step
maizecornleaves.jfif maize maize True
1/1 [==============================] - 0s 81ms/step
rice-5174887_1280.jpg maize rice False
1/1 [==============================] - 0s 81ms/step
maize-field01.jpg wheat maize False
1/1 [==============================] - 0s 80ms/step
juteimg.jpg rice jute False
1/1 [==============================] - 0s 81ms/step
rice-828540_1280.jpg rice rice True
1/1 [==============================] - 0s 80ms/step
wheat-crop.jpg maize wheat False
1/1 [===================

In [ ]:
acc_resnet = sum(preds_resnet)/len(preds_resnet)
acc_resnet

0.2549019607843137

In [ ]:
preds_inception = []
for file in filenames:
    yp = data[get_pred(file, model_inception)]
    ya = get_class_name(file)
    pred = (yp == ya)
    preds_inception.append(pred)
    print(file, yp, ya, pred)

1/1 [==============================] - 1s 1s/step
maize corn set2.jpg maize maize True
1/1 [==============================] - 0s 59ms/step
wheat-field.jfif sugarcane wheat False
1/1 [==============================] - 0s 59ms/step
wheatcropfield04.jpg sugarcane wheat False
1/1 [==============================] - 0s 60ms/step
maize00corn-fields.jpg maize maize True
1/1 [==============================] - 0s 58ms/step
maize-Field-Corn.jpg maize maize True
1/1 [==============================] - 0s 58ms/step
maizecornleaves.jfif sugarcane maize False
1/1 [==============================] - 0s 58ms/step
rice-5174887_1280.jpg wheat rice False
1/1 [==============================] - 0s 57ms/step
maize-field01.jpg maize maize True
1/1 [==============================] - 0s 59ms/step
juteimg.jpg rice jute False
1/1 [==============================] - 0s 58ms/step
rice-828540_1280.jpg maize rice False
1/1 [==============================] - 0s 56ms/step
wheat-crop.jpg wheat wheat True
1/1 [=============

In [ ]:
acc_inception = sum(preds_inception)/len(preds_inception)
acc_inception

0.45098039215686275

In [ ]:
result = pd.DataFrame(data = {'Sequential': acc_model, 'VGG16': acc_vgg, 'Resnet': acc_resnet, 'Inception': acc_inception}, index=['Accuracy'])
result

,Sequential,VGG16,Resnet,Inception
Accuracy,0.176471,0.490196,0.254902,0.45098
